In [1]:
import os

import numpy as np
import pandas as pd

In [2]:
DIRECTORY = './data/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.dat')]
dataset_list.sort()

In [3]:
dataset_list

['CR1000_Table1_2020_05_06_21_19_15.dat',
 'CR1000_Table1_2020_05_13_21_19_15.dat',
 'CR1000_Table1_2020_05_25_14_35_46.dat',
 'CR1000_Table1_2020_06_10_16_29_39.dat',
 'CR1000_Table1_2020_06_18_15_51_17.dat',
 'CR1000_Table1_2020_07_13_15_19_58.dat']

In [4]:
greenhouse_df = []
for FILENAME in dataset_list:
    temp_df = pd.read_csv(DIRECTORY + FILENAME, sep=',', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    greenhouse_df.append(temp_df)
greenhouse_df = pd.concat(greenhouse_df)

/home/phil/.virtualenvs/tf20/lib/python3.6/site-packages/ipykernel_launcher.py:5: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  """


In [5]:
greenhouse_df.index = pd.DatetimeIndex(greenhouse_df.index)

In [6]:
cr1000_df = greenhouse_df
cr1000_df = cr1000_df[['Loadcell_1', 'Loadcell_2', 'Pyrano_Wsec_1', 'Temp_Avg', 'Humid_Avg']]

In [7]:
date_index = pd.date_range(cr1000_df.index[0], cr1000_df.index[-1], freq='1 min')

In [8]:
cr1000_df = cr1000_df.reindex(date_index)

In [9]:
DIRECTORY = './data/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [10]:
greenhouse_df = []
divider = 0
for FILENAME in dataset_list:
    if 'A2' in FILENAME:
        divider += 1
    temp_df = pd.read_excel(DIRECTORY + FILENAME, index_col='date')
    temp_df.index = pd.DatetimeIndex(temp_df.index)
    temp_df = temp_df[['weight', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']]
    greenhouse_df.append(temp_df)

In [11]:
for i in range(len(greenhouse_df)):
    if greenhouse_df[i].index.round('1 min')[0].minute == 1:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('1 min') - pd.Timedelta('1 min'))
    elif greenhouse_df[i].index.round('1 min')[0].minute == 2:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('2 min') - pd.Timedelta('2 min'))
    else:
        greenhouse_df[i].index = greenhouse_df[i].index.round('1 min')

In [12]:
for i in range(len(greenhouse_df)):
    greenhouse_df[i] = greenhouse_df[i].groupby(greenhouse_df[i].index).first()

In [13]:
date_index = pd.date_range(greenhouse_df[0].index[0], greenhouse_df[-1].index[-1], freq='1 min')

In [14]:
_1 = pd.concat(greenhouse_df[:divider]).reindex(date_index)
_2 = pd.concat(greenhouse_df[divider:]).reindex(date_index)

In [15]:
_1.columns = ['Loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']
_2.columns = ['Loadcell_4', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']

In [16]:
SW2_df = pd.concat([_1.loc[pd.date_range(cr1000_df.index[0], _1.index[-1], freq='1 min')],
                    cr1000_df.loc[pd.date_range(cr1000_df.index[0], _1.index[-1], freq='1 min')]], axis=1)

In [17]:
SW2_df.columns = ['loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'rad', 'temp', 'hum']

In [18]:
SW2_df = SW2_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'loadcell_3']]

In [19]:
SW2_df['temp'] = ((SW2_df['temp']-1000)/80)
SW2_df['hum'] = ((SW2_df['hum']-1000)/40)
SW2_df['loadcell_1'] = ((SW2_df['loadcell_1']/100))
SW2_df['loadcell_2'] = ((SW2_df['loadcell_2']/100))

In [20]:
SW2_df.to_csv('./results/SW2_greenhouse.csv')